# CS781 capstone - Phase 1: Legal language model (ROPEFIX_295M)
Donkena Manideep (261040623).  Runs the code of `phase1/` from my submission on Kaggle.

**Before running:** turn on a GPU (T4 x2 or better; the code uses bfloat16), turn on Internet (for the weights and
`pip`), and attach as Kaggle datasets: (a) the code (the `code/` folder of my archive) unless the GitHub repository is
public, (b) the Phase 1 competition data (not needed: only the trained model is used here).
Set `DATA_INPUT` below to the folder of (b).  `QUICK = True` runs short smoke tests; `False` runs the full recipe
(the full runs took 15-95 minutes each on an RTX PRO 5000 and take several times longer on a T4).

In [ ]:
PHASE = 1
QUICK = True                                    # False = the full recipe
DATA_INPUT = "/kaggle/input/cs781-phase1-data/Phase_1_Pretraining"   # <- the attached competition data
DATA_SUBDIR = "Phase_1_Pretraining"
REPO_URL = "https://github.com/ManideepDonkena/CS781-Legal-Intelligence-System"
REL = "https://github.com/ManideepDonkena/CS781-Legal-Intelligence-System/releases/download/models-v1/"

In [ ]:
# ---- setup: code, weights, folder layout ----------------------------------------------------------------------
import glob, os, shutil, subprocess, sys, urllib.request

def sh(cmd):
    """Run a shell command, print its output as it comes, stop on error."""
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if r.returncode:
        raise RuntimeError(f"failed ({r.returncode}): {cmd}")

# 1. the code: an attached Kaggle dataset that holds phase1..phase7 (the unzipped archive's code/ folder),
#    else a clone of the GitHub repository (the repository must be public for this)
found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/phase1/final_model", recursive=True)]
if found:
    REPO = os.path.dirname(found[0])
else:
    REPO = "/kaggle/working/repo"
    if not os.path.isdir(REPO):
        sh(f"git clone --depth 1 {REPO_URL} {REPO}")
print("code:", REPO)
sh(f"pip install -q -r {REPO}/requirements.txt || true")

# 2. the layout the code expects
CAP = "/home/mani/cs781/Capstone"
LLM = "/home/mani/cs781_phase1/Legal_LLM"
os.makedirs(os.path.dirname(LLM), exist_ok=True)
if not os.path.exists(LLM):
    shutil.copytree(f"{REPO}/phase1/final_model", LLM)                 # model code (modern_arch.py, rope.py, ...)
NAMES = {2: "Phase2_LNER", 3: "Phase3_RR", 4: "Phase4_CJPE", 5: "Phase5_LSI", 6: "Phase6_SUM", 7: "Phase7_PCR"}
for p, name in NAMES.items():                                          # every phase's src (they import each other)
    os.makedirs(f"{CAP}/{name}/runs", exist_ok=True)
    if not os.path.exists(f"{CAP}/{name}/src"):
        shutil.copytree(f"{REPO}/phase{p}/src", f"{CAP}/{name}/src")
if not os.path.exists(f"{CAP}/shared"):
    shutil.copytree(f"{REPO}/shared", f"{CAP}/shared")

# 3. this phase's competition data (attach it as a Kaggle dataset; DATA_INPUT is its folder)
if PHASE > 1:
    target = f"{CAP}/{NAMES[PHASE]}/data/{DATA_SUBDIR}"
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.exists(target):
        os.symlink(DATA_INPUT, target)
    print("data:", target, "->", os.listdir(target)[:10])

# 4. the Phase 1 weights from the GitHub release models-v1 (about 1.2 GB)
W = "/workspace/models/cs781_phase1_runs/ROPEFIX_295M"
os.makedirs(W, exist_ok=True)
for name, dest in [("phase1_ROPEFIX_295M_weights.pt", "best.pt"), ("phase1_tokenizer.json", "tokenizer.json"),
                   ("phase1_run_config.json", "run_config.json")]:
    if not os.path.exists(f"{W}/{dest}"):
        print("downloading", name, flush=True)
        urllib.request.urlretrieve(REL + name, f"{W}/{dest}")
sys.path[:0] = [LLM, f"{CAP}/shared"]
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
      "| bf16:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

## The model and the code it uses
The final model's code is `phase1/final_model/` (`modern_arch.py` uses `rope.py` and `attention.py` from my Assignment 3 unchanged).

In [ ]:
# ---- the Phase 1 model loads and uses my Assignment 3 code ------------------------------------------------------
sys.path.insert(0, f"{CAP}/Phase4_CJPE/src")
from ft_doc import load_model
import modern_arch, rope, attention
lm, tok, family = load_model(W, LLM)
print(type(lm).__name__, "| attention:", type(lm.layers[0].attn).__mro__[1].__module__ + ".CausalSelfAttention",
      "| RoPE:", rope.build_rope_cache.__module__ + ".build_rope_cache")
print(f"{sum(p.numel() for p in lm.parameters()):,} parameters")

In [ ]:
# ---- loss on a few sentences and a short continuation (greedy) --------------------------------------------------
lm = lm.cuda().eval()
text = "The appellant filed a writ petition under Article 226 of the Constitution challenging the order of the"
ids = torch.tensor([tok.encode(text)]).cuda()
with torch.no_grad(), torch.autocast("cuda", dtype=torch.bfloat16):
    print("loss per token:", lm(ids, labels=ids)["loss"].item())
    for _ in range(40):
        nxt = lm(ids)["logits"][0, -1].argmax()
        ids = torch.cat([ids, nxt.view(1, 1)], 1)
print(tok.decode(ids[0].tolist()))

## Training (needs the Phase 1 data cache; 38.7 GPU-hours for the full model)
The exact command that trained ROPEFIX_295M is below; with `QUICK` it trains a small model for 20M tokens only to show the program runs. See `phase1/final_model/README.md`.

In [ ]:
cmd = (f"cd {LLM} && python studies/train_job.py --id ROPEFIX_295M --study ROPEFIX --cache final_bpe32k_mixA_3B "
       "--d-model 1024 --layers 23 --heads 16 --tokens 3000000000 --bs 8 --ga 4 --lr 1e-3 --wd 0.1 --seed 42 "
       "--extra '{\"rope_float\": true}' --keep-best")
if QUICK:
    cmd = cmd.replace("--d-model 1024 --layers 23", "--d-model 256 --layers 4").replace("3000000000", "20000000")
print(cmd)   # needs the token cache built by studies/build_cache.py from the Phase 1 data; run it once that exists:
# sh(cmd)